In [1]:

# ============================================
# PROYEK DATA MINING - KUALITAS UDARA
# UJI API OPEN-METEO: KOTA MEDAN
# ============================================

import requests
import pandas as pd

# 1. Alamat API
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

# 2. Koordinat Kota Medan
latitude = 3.5952
longitude = 98.6722

# 3. Parameter permintaan data
params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": (
        "pm2_5,pm10,carbon_monoxide,"
        "nitrogen_dioxide,sulphur_dioxide,ozone"
    ),
    "domains": "cams_global",
    "past_days": 90,
    "forecast_days": 0,
    "timezone": "Asia/Jakarta"
}

# 4. Mengirim permintaan ke API
try:
    response = requests.get(
        url,
        params=params,
        timeout=60
    )

    response.raise_for_status()
    data = response.json()

    if data.get("error"):
        raise ValueError(data.get("reason", "API mengembalikan error"))

    print("BERHASIL TERHUBUNG KE OPEN-METEO API")

except requests.exceptions.RequestException as error:
    print("GAGAL MENGAMBIL DATA:", error)
    raise

# 5. Mengambil data per jam
hourly = data.get("hourly", {})

if not hourly:
    raise ValueError("Data hourly tidak ditemukan.")

# 6. Mengubah data menjadi DataFrame
df = pd.DataFrame(hourly)

# Pastikan keenam variabel tersedia
fitur = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

kolom_hilang = [kolom for kolom in fitur if kolom not in df.columns]

if kolom_hilang:
    raise ValueError(f"Variabel tidak ditemukan: {kolom_hilang}")

# 7. Menambahkan informasi lokasi
df.insert(0, "kota", "Medan")
df.insert(1, "provinsi", "Sumatera Utara")

# 8. Menyimpan koordinat yang diminta
df["requested_latitude"] = latitude
df["requested_longitude"] = longitude

# Koordinat pusat grid model yang benar-benar digunakan API
df["model_grid_latitude"] = data.get("latitude")
df["model_grid_longitude"] = data.get("longitude")

# 9. Memeriksa waktu pengamatan
df["time"] = pd.to_datetime(df["time"])

# 10. Menampilkan ringkasan dataset
print("\n========== INFORMASI DATASET ==========")
print("Kota:", "Medan")
print("Zona waktu:", data.get("timezone"))
print("Resolusi data:", "CAMS Global - 3 jam")
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Tanggal awal:", df["time"].min())
print("Tanggal akhir:", df["time"].max())
print("Jumlah tanggal:", df["time"].dt.date.nunique())

print("\nKoordinat yang diminta:")
print(latitude, longitude)

print("\nKoordinat grid model dari API:")
print(data.get("latitude"), data.get("longitude"))

print("\nSatuan variabel:")
print(data.get("hourly_units"))

print("\nPemeriksaan data kosong:")
print(df[fitur].isna().sum())

print("\nPemeriksaan duplikasi waktu:")
print(df["time"].duplicated().sum())

# 11. Melihat 10 baris pertama
print("\n========== CONTOH DATA ==========")
display(df.head(10))

# 12. Statistik deskriptif
print("\n========== STATISTIK DESKRIPTIF ==========")
display(df[fitur].describe())

# 13. Menyimpan dataset mentah
nama_file = "uji_api_medan_90_hari.csv"

df.to_csv(nama_file, index=False)

print("\nDataset berhasil disimpan:", nama_file)

# 14. Mengunduh CSV ke laptop
try:
    from google.colab import files
    files.download(nama_file)
except ImportError:
    print("File tersimpan di lingkungan kerja saat ini.")


BERHASIL TERHUBUNG KE OPEN-METEO API

========== INFORMASI DATASET ==========
Kota: Medan
Zona waktu: Asia/Jakarta
Resolusi data: CAMS Global - 3 jam
Jumlah baris: 2160
Jumlah kolom: 13
Tanggal awal: 2026-08-20 00:00:00
Tanggal akhir: 2026-11-17 23:00:00
Jumlah tanggal: 90

Koordinat yang diminta:
3.5952 98.6722

Koordinat grid model dari API:
3.5999985 98.70001

Satuan variabel:
{'time': 'iso8601', 'pm2_5': 'μg/m³', 'pm10': 'μg/m³', 'carbon_monoxide': 'μg/m³', 'nitrogen_dioxide': 'μg/m³', 'sulphur_dioxide': 'μg/m³', 'ozone': 'μg/m³'}

Pemeriksaan data kosong:
pm2_5               0
pm10                0
carbon_monoxide     0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
dtype: int64

Pemeriksaan duplikasi waktu:
0

========== CONTOH DATA ==========


,kota,provinsi,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,requested_latitude,requested_longitude,model_grid_latitude,model_grid_longitude
0,Medan,Sumatera Utara,2026-08-20 00:00:00,41.9,43.5,1664.0,38.1,10.8,16.0,3.5952,98.6722,3.599998,98.70001
1,Medan,Sumatera Utara,2026-08-20 01:00:00,39.7,41.3,1585.0,36.7,9.8,12.0,3.5952,98.6722,3.599998,98.70001
2,Medan,Sumatera Utara,2026-08-20 02:00:00,37.4,38.8,1369.0,34.0,8.8,8.0,3.5952,98.6722,3.599998,98.70001
3,Medan,Sumatera Utara,2026-08-20 03:00:00,36.3,37.4,1096.0,30.6,7.8,4.0,3.5952,98.6722,3.599998,98.70001
4,Medan,Sumatera Utara,2026-08-20 04:00:00,35.2,36.0,918.0,28.0,7.3,3.0,3.5952,98.6722,3.599998,98.70001
5,Medan,Sumatera Utara,2026-08-20 05:00:00,34.0,34.7,913.0,27.8,7.6,0.0,3.5952,98.6722,3.599998,98.70001
6,Medan,Sumatera Utara,2026-08-20 06:00:00,32.6,33.2,1003.0,28.4,8.2,0.0,3.5952,98.6722,3.599998,98.70001
7,Medan,Sumatera Utara,2026-08-20 07:00:00,38.9,39.1,1104.0,31.0,8.5,8.0,3.5952,98.6722,3.599998,98.70001
8,Medan,Sumatera Utara,2026-08-20 08:00:00,41.7,42.5,1231.0,26.7,9.4,29.0,3.5952,98.6722,3.599998,98.70001
9,Medan,Sumatera Utara,2026-08-20 09:00:00,36.9,37.7,1369.0,20.8,10.7,59.0,3.5952,98.6722,3.599998,98.70001



========== STATISTIK DESKRIPTIF ==========


,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone
count,2160.000000,2160.000000,2160.000000,2160.000000,2160.000000,2160.000000
mean,39.547824,40.455324,1805.717130,24.629028,10.263056,71.643519
std,23.823805,24.317582,1169.635328,18.033058,4.622359,59.419024
min,6.100000,6.200000,244.000000,0.700000,2.900000,0.000000
25%,21.075000,21.500000,970.000000,9.475000,7.100000,20.000000
50%,35.950000,36.700000,1521.000000,22.100000,9.200000,52.000000
75%,51.025000,52.300000,2342.000000,34.000000,12.100000,122.000000
max,165.800000,170.000000,8643.000000,100.300000,34.300000,239.000000



Dataset berhasil disimpan: uji_api_medan_90_hari.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>